# 05 - Events and profiles

Explores the two parquet files produced by the intake step in `out/data/`:

| File | Grain | Columns |
|---|---|---|
| `events.parquet` | one row per event | `user_id`, `ts`, `source`, `fields` (map of string to string) |
| `profiles.parquet` | one row per customer | `user_id`, `as_of`, `attributes` (map), `lifelong` (list of key/timestamp) |

`events.parquet` has ~12.9M rows, so the notebook reads only the columns and row groups it needs instead of loading the whole file.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "out" / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "out" / "data"
EVENTS, PROFILES = DATA / "events.parquet", DATA / "profiles.parquet"
print(DATA)
for p in sorted(DATA.iterdir()):
    print(f"{p.name:25s} {p.stat().st_size / 1e6:10.1f} MB")

## 1. Intake reports

In [ ]:
out_rep = json.loads((DATA / "output_report.json").read_text())
print("generated      :", out_rep["generated"])
print("rows in        :", f"{out_rep['rows_in']:,}")
print("events out     :", f"{out_rep['events_out']:,}")
print("customers      :", f"{out_rep['customers']:,}")
print("profiles out   :", f"{out_rep['profiles_out']:,}")
print("vocab estimate :", f"{out_rep['vocabulary_estimate']:,}")
pd.Series(out_rep["by_source"], name="events").to_frame()

In [ ]:
# Per source table: rows in, events kept, and why rows were dropped
rows = []
for table, r in out_rep["reconciliation"].items():
    rows.append({"table": table, "rows_in": r["rows_in"], "events": r["events"],
                 "dropped": sum(r["dropped"].values()), "unexplained": r["unexplained"],
                 "reasons": "; ".join(f"{k} ({v:,})" for k, v in r["dropped"].items())})
pd.DataFrame(rows).set_index("table")

In [ ]:
qual = json.loads((DATA / "quality_report.json").read_text())
print(f"client: {qual['client']}   readiness: {qual['readiness']['score']} ({qual['readiness']['band']})")
checks = pd.DataFrame(qual["readiness"]["checks"])
checks["n_findings"] = checks["findings"].str.len()
checks[["name", "weight", "scored", "n_findings"]]

In [ ]:
for c in qual["readiness"]["checks"]:
    for f in c["findings"]:
        print(f"[{c['name']}] {f}")

## 2. Events

In [ ]:
ev_file = pq.ParquetFile(EVENTS)
print("rows       :", f"{ev_file.metadata.num_rows:,}")
print("row groups :", ev_file.metadata.num_row_groups)
print(ev_file.schema_arrow)

In [ ]:
# A small sample from the first row group. Used for the field-level views below.
sample = ev_file.read_row_group(0).to_pandas()
print(sample.shape)
sample.head()

In [ ]:
# Cheap full-file aggregates: read one column at a time
src = ev_file.read(columns=["source"]).column("source").value_counts().to_pylist()
src = pd.DataFrame(src).rename(columns={"values": "source", "counts": "events"}).set_index("source")
src["share"] = (src["events"] / src["events"].sum()).round(4)
src

In [ ]:
# Time range from the ts column
ts = ev_file.read(columns=["ts"]).column("ts")
print("first event:", pc.min(ts).as_py())
print("last event :", pc.max(ts).as_py())

In [ ]:
# Events per customer
per_user = ev_file.read(columns=["user_id"]).column("user_id").value_counts().to_pylist()
per_user = (pd.DataFrame(per_user).rename(columns={"values": "user_id", "counts": "n_events"})
            .set_index("user_id")["n_events"])
print("customers with events:", f"{len(per_user):,}")
per_user.describe(percentiles=[.01, .1, .5, .9, .99]).round(1)

In [ ]:
# Which keys live in the `fields` map, per source (from the sample row group)
def field_keys(df):
    return df["fields"].map(lambda kv: [k for k, _ in kv]).explode().value_counts()

for s, g in sample.groupby("source"):
    keys = field_keys(g)
    print(f"--- {s}: {len(g):,} events, {len(keys)} distinct keys")
    print((keys / len(g)).round(3).head(15).to_string())

In [ ]:
# Look at one source with `fields` expanded into columns
def expand(df):
    wide = pd.DataFrame([dict(kv) for kv in df["fields"]], index=df.index)
    return pd.concat([df[["user_id", "ts", "source"]], wide], axis=1)

expand(sample[sample["source"] == "transaction"].head(10)).dropna(axis=1, how="all")

In [ ]:
# One customer's timeline (first 25 events)
uid = sample["user_id"].iloc[0]
tl = pq.read_table(EVENTS, filters=[("user_id", "=", uid)]).to_pandas().sort_values("ts")
print(uid, f"- {len(tl):,} events, {tl['ts'].min()} -> {tl['ts'].max()}")
tl.assign(fields=tl["fields"].map(dict)).head(25)

## 3. Profiles

In [ ]:
prof = pq.read_table(PROFILES).to_pandas()
print(prof.shape, "| unique customers:", prof["user_id"].nunique())
print("as_of:", prof["as_of"].min(), "->", prof["as_of"].max())
prof.head()

In [ ]:
# Expand the `attributes` map into one column per attribute
attrs = pd.DataFrame([dict(kv) for kv in prof["attributes"]], index=prof["user_id"])
print(attrs.shape)
fill = attrs.notna().mean().sort_values(ascending=False).rename("filled").to_frame()
fill["distinct"] = attrs.nunique()
fill

In [ ]:
# Distribution of a few low cardinality attributes
low = [c for c in attrs.columns if attrs[c].nunique() <= 8][:6]
for c in low:
    print(attrs[c].value_counts(dropna=False).to_string(), "\n")

In [ ]:
# `lifelong` holds dated facts that never change (open dates, KYC reviews, ...)
life = prof[["user_id", "lifelong"]].explode("lifelong").dropna(subset=["lifelong"]).reset_index(drop=True)
life = pd.concat([life["user_id"], pd.json_normalize(life["lifelong"].tolist())], axis=1)
print(life.shape)
life.groupby("key")["ts"].agg(["count", "min", "max"])

## 4. Events joined to profiles

In [ ]:
# Do the two files cover the same customers?
ev_users, pr_users = set(per_user.index), set(prof["user_id"])
print("customers in events  :", f"{len(ev_users):,}")
print("customers in profiles:", f"{len(pr_users):,}")
print("in both              :", f"{len(ev_users & pr_users):,}")
print("events but no profile:", f"{len(ev_users - pr_users):,}")
print("profile but no events:", f"{len(pr_users - ev_users):,}")

In [ ]:
# Event volume by customer segment, if the attribute exists
if "cust_segment" in attrs.columns:
    j = attrs[["cust_segment"]].join(per_user.rename("n_events")).fillna({"n_events": 0})
    display(j.groupby("cust_segment")["n_events"].agg(["count", "mean", "median", "max"]).round(1))